In [ ]:
# --- config ---
DATASET = "git_linux"  # transactions | senedd | git_linux | git_pytorch

In [ ]:
# --- load ---
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from presets import PRESETS

preset = PRESETS[DATASET]

folder = Path("../outputs") / DATASET
manifest = json.loads((folder / "manifest.json").read_text())
embeddings = np.load(folder / "embeddings.npy")
metadata = json.loads((folder / "metadata.json").read_text())

print(manifest)

model = SentenceTransformer(manifest["model_name"])
FIELDS = preset["fields"]
LABEL_FIELD = preset["label_field"]


def describe(i, width=120):
    """One line per item: its label, then the text that was embedded."""
    m = metadata[i]
    text = " ".join(str(m.get(f, "")) for f in FIELDS) if FIELDS else str(m)
    label = f"[{m.get(LABEL_FIELD, '')}] " if LABEL_FIELD else ""
    return (label + text)[:width]


print(f"{DATASET}: {len(embeddings)} embeddings from {manifest['model_name']}")

In [ ]:
# --- semantic search ---
import numpy as np

def search(query, top_k=5):
    q = model.encode(query, normalize_embeddings=True)
    scores = np.dot(embeddings, q)   # cosine similarity, since embeddings are normalised
    best = np.argsort(-scores)[:top_k]
    return [(float(scores[i]), metadata[i]) for i in best]

for score, record in search("new nvidia kernel"):
    print(f"{score:.3f}  {record}")

In [ ]:
# --- clustering ---
from sklearn.cluster import KMeans

N_CLUSTERS = 5   # change this and re-run

labels = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(embeddings)

for cluster_id in range(N_CLUSTERS):
    items = [metadata[i] for i in range(len(labels)) if labels[i] == cluster_id][:5]
    print(f"\ncluster {cluster_id}:")
    for item in items:
        print(" ", item)

In [ ]:
# --- 2D projections: PCA and t-SNE ---
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

PERPLEXITY = 30   # t-SNE: roughly how many neighbours matter; try 5, 15, 50

def pca_projection(embeddings):
    return PCA(n_components=2).fit_transform(embeddings)

def tsne_projection(embeddings, perplexity=PERPLEXITY):
    return TSNE(n_components=2, perplexity=perplexity, metric="cosine",
                init="pca", random_state=42).fit_transform(embeddings)

def plot_projection(coords, labels, title):
    plt.figure(figsize=(8, 6))
    plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=20)
    plt.title(f"{title}: {manifest['modality']} embeddings, {N_CLUSTERS} clusters")
    plt.show()

plot_projection(pca_projection(embeddings), labels, "PCA")
plot_projection(tsne_projection(embeddings), labels, "t-SNE")

In [ ]:
# --- community discovery (no fixed k - the graph decides) ---
from collections import Counter
import networkx as nx
import numpy as np
import pandas as pd
from presets import PRESETS

SIMILARITY_THRESHOLD = 0.5   # change this and re-run
MIN_SIZE = 5                 # ignore communities smaller than this

FIELDS = PRESETS[DATASET]["fields"]
LABEL = PRESETS[DATASET]["label_field"]

def short(i, width=80):
    return " ".join(str(metadata[i][f]) for f in FIELDS)[:width]

# connect every pair of items more similar than the threshold
sims = np.dot(embeddings, embeddings.T)
rows, cols = np.nonzero(np.triu(sims, k=1) > SIMILARITY_THRESHOLD)
graph = nx.Graph()
graph.add_nodes_from(range(len(embeddings)))
graph.add_weighted_edges_from(zip(rows.tolist(), cols.tolist(), sims[rows, cols].tolist()))

communities = [sorted(c) for c in nx.community.louvain_communities(graph, seed=42)
               if len(c) >= MIN_SIZE]
communities.sort(key=len, reverse=True)

# one row per community: size, dominant label, and its most typical member
centroids = np.array([embeddings[members].mean(axis=0) for members in communities])
summary = []
for idx, (members, centre) in enumerate(zip(communities, centroids)):
    typical = members[int(np.argmax(np.dot(embeddings[members], centre)))]
    row = {"community": idx, "size": len(members), "typical member": short(typical)}
    if LABEL:
        label, count = Counter(metadata[i][LABEL] for i in members).most_common(1)[0]
        row["top label"] = f"{label} ({count / len(members):.0%})"
    summary.append(row)

print(f"{graph.number_of_edges()} edges, {len(communities)} communities of {MIN_SIZE}+ items")
display(pd.DataFrame(summary).set_index("community"))

In [ ]:
# --- community tree: which communities are most alike? ---
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

tree = linkage(centroids, method="average", metric="cosine")
leaf_names = [f"{row['community']}: {row['typical member'][:50]} ({row['size']})"
              for row in summary]

plt.figure(figsize=(10, max(4, 0.3 * len(communities))))
dendrogram(tree, orientation="right", labels=leaf_names, leaf_font_size=9)
plt.title(f"{DATASET}: communities grouped by similarity")
plt.tight_layout()
plt.show()